# Crop Yield + Climate Fine-Tuning (Mistral 7B)

Fine-tunes Mistral-7B-Instruct on a crop-climate dataset using QLoRA via Unsloth. Given country, year, CO2 level, temperature anomaly, and crop list, the model generates a structured agricultural analysis.

**Before running:**
1. Runtime → Change runtime type → GPU → T4
2. Open the Secrets panel (🔑 left sidebar) and add:
   - `HF_TOKEN` — your token from https://huggingface.co/settings/tokens
   - `HF_USERNAME` — your Hugging Face username
3. Upload `train.jsonl` and `val.jsonl` when prompted in cell 4
4. Run cells top to bottom

| Step | Approx. time on free T4 |
|------|--------------------------|
| Install packages | 3–5 min |
| Load model | 3–4 min |
| Dataset prep | < 1 min |
| Training (2 epochs) | 55–75 min |
| Push to Hub | 25–35 min |


In [1]:
# 1. Install dependencies
# Unsloth pins specific torch/xformers versions so let it handle resolution
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes datasets huggingface_hub -q
print('packages ready')


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.5 MB/s eta 0:00:00
packages ready


In [2]:
# 2. Check GPU
import torch
assert torch.cuda.is_available(), 'No GPU — go to Runtime > Change runtime type > GPU > T4'
print(torch.cuda.get_device_name(0))
print(f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM')


Tesla T4
15.6 GB VRAM


In [4]:
# 3. Load credentials from Colab Secrets
from google.colab import userdata

HF_TOKEN    = userdata.get('HF_TOKEN')
HF_USERNAME = userdata.get('HF_USERNAME')
REPO_NAME   = 'crop-climate-mistral-7b'

assert HF_TOKEN,    'Add HF_TOKEN to Colab Secrets'
assert HF_USERNAME, 'Add HF_USERNAME to Colab Secrets'
print('credentials loaded')


credentials loaded


In [6]:
# 4. Get dataset files
# Option A: drag train.jsonl and val.jsonl into the Files panel on the left

# Option B: pull from a HuggingFace dataset repo
# from huggingface_hub import login, hf_hub_download
# login(token=HF_TOKEN)
# for fname in ['train.jsonl', 'val.jsonl']:
#     hf_hub_download(
#         repo_id=f'{HF_USERNAME}/crop-climate-data',
#         filename=fname,
#         repo_type='dataset',
#         local_dir='.'
#     )

import os
assert os.path.exists('train.jsonl'), 'train.jsonl not found — upload it first'
assert os.path.exists('val.jsonl'),   'val.jsonl not found — upload it first'
print('dataset files found')


dataset files found


In [7]:
# 5. Load Mistral 7B in 4-bit (QLoRA) — fits in T4's 15 GB VRAM
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit',
    max_seq_length = 512,
    dtype          = None,
    load_in_4bit   = True,
)
print('model loaded')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

model loaded


In [8]:
# 6. Attach LoRA adapters
# r=16 gives a good quality/VRAM tradeoff on T4
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                       'gate_proj','up_proj','down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()


Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


In [9]:
# 7. Load and format dataset
from datasets import load_dataset

raw = load_dataset('json', data_files={
    'train':      'train.jsonl',
    'validation': 'val.jsonl',
})

def format_chat(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}

dataset = raw.map(format_chat, batched=True, remove_columns=['messages'])

# drop examples longer than max_seq_length to avoid truncation surprises
MAX_TOKENS = 512
def count_tokens(examples):
    return {'n_tokens': [len(tokenizer.encode(t)) for t in examples['text']]}

dataset = dataset.map(count_tokens, batched=True)

for split in ['train', 'validation']:
    before = len(dataset[split])
    dataset[split] = dataset[split].filter(lambda x: x['n_tokens'] <= MAX_TOKENS)
    after  = len(dataset[split])
    print(f'{split}: {after} kept, {before - after} dropped (>{MAX_TOKENS} tokens)')

print()
print('Sample (first 300 chars):')
print(dataset['train'][0]['text'][:300])


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8460 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

Map:   0%|          | 0/8460 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8460 [00:00<?, ? examples/s]

train: 1148 kept, 7312 dropped (>512 tokens)


Filter:   0%|          | 0/941 [00:00<?, ? examples/s]

validation: 104 kept, 837 dropped (>512 tokens)

Sample (first 300 chars):
<s>[INST] You are an agricultural analyst. Given the following data for Guyana in 1981, analyze the crop yields and climate context.

Country: Guyana
Year: 1981 (1980s)
Global CO2: 340.12 ppm (Early industrial era (320–350 ppm))
Temperature anomaly: 0.32°C (Moderately warm)
Population: 782,000
Overa


In [10]:
# 8. Configure trainer
# SFTConfig is required here — TrainingArguments causes a PicklingError
# when Unsloth's compiled trainer tries to save checkpoints
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = './crop-climate-mistral',
    num_train_epochs            = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,       # effective batch size = 8
    warmup_steps                = 50,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 25,
    eval_strategy               = 'steps',
    eval_steps                  = 200,
    save_strategy               = 'steps',
    save_steps                  = 200,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    seed                        = 42,
    report_to                   = 'none',
    dataloader_pin_memory       = False,
    dataset_text_field          = 'text',
    max_seq_length              = 512,
    packing                     = False,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset  = dataset['validation'],
    args          = training_args,
)
print('trainer ready')


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1148 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/104 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
trainer ready


In [11]:
# 9. Train
# ~55-75 min on free T4 — keep the browser tab active
trainer_stats = trainer.train()
print('training done')
print(f'  loss:    {trainer_stats.metrics["train_loss"]:.4f}')
print(f'  runtime: {trainer_stats.metrics["train_runtime"]/60:.1f} min')


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,148 | Num Epochs = 1 | Total steps = 287
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
200,0.112433,1.192508
287,0.100123,1.181418


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./crop-climate-mistral/checkpoint-200.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`Attent

training done
  loss:    0.2086
  runtime: 48.6 min


In [12]:
# 10. Quick inference test
FastLanguageModel.for_inference(model)

test_messages = [{
    'role': 'user',
    'content': (
        'You are an agricultural analyst. Given the following data for India '
        'in 2015, analyze the crop yields and climate context.\n\n'
        'Country: India\nYear: 2015 (2010s)\n'
        'Global CO2: 401.00 ppm (Critical CO2 era (> 400 ppm))\n'
        'Temperature anomaly: 0.90°C (Warm year)\n'
        'Population: 1,309,000,000\n\n'
        'Crops recorded this year: wheat, rice, maize, potatoes\n\n'
        'Provide a structured agricultural analysis including yield performance, '
        'climate impact, and key observations.'
    )
}]

prompt = tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True
)
inputs  = tokenizer([prompt], return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    temperature    = 0.7,
    do_sample      = True,
    pad_token_id   = tokenizer.eos_token_id,
)
generated = outputs[0][inputs['input_ids'].shape[-1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

## Agricultural Analysis: India, 2015

### Crop Yield Performance
- **Wheat**: 2.54 t/ha — good
- **Rice**: 3.01 t/ha — good
- **Maize**: 2.90 t/ha — good
- **Potatoes**: 23.28 t/ha — excellent

### Climate Context
- CO2 concentration: 401.00 ppm (Critical CO2 era (> 400 ppm))
- Temperature anomaly: 0.90°C above baseline (Warm year)

### Key Observations
- 2015 was an extremely warm year globally (+0.90°C anomaly), likely contributing to heat stress in temperature-sensitive crops.
- Atmospheric CO2 exceeded 400 ppm (401.00 ppm), a level associated with measurable yield impacts and accelerating climate change pressure on agriculture.
- Strongest performer: **Potatoes** at 23.28 t/ha (excellent).
- Most constrained: **Wheat** at 2.54 t/ha (good).

### Historical Context
This data point is from the **2010s**, when global CO2 was in the **critical co2 era (> 400 ppm)**. Agricultural technology and climate pressure were rapidly evolving.


In [14]:
# 11. Push merged model to Hugging Face Hub
# This merges the LoRA weights back into the base model and uploads the full 16-bit model
from huggingface_hub import login
login(token=HF_TOKEN)

model.push_to_hub_merged(
    f'{HF_USERNAME}/{REPO_NAME}',
    tokenizer,
    save_method = 'merged_16bit',
    token = HF_TOKEN,
)
print(f'model live at: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 3/3 [00:00<00:00, 18921.67it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [04:27<00:00, 89.05s/it]


Unsloth: Merge process complete. Saved to `/content/AliBuxdev/crop-climate-mistral-7b`
model live at: https://huggingface.co/AliBuxdev/crop-climate-mistral-7b


## Done

The fine-tuned model is on Hugging Face. A few things worth doing next:
- Write a model card on HF describing the dataset, training setup, and example outputs
- Link the repo from your GitHub README
- Step 4: wrap this in a FastAPI endpoint
